In [1]:
import os
from functools import partial
from scipy import constants
import joblib
import sys
import jax
import jax.numpy as jnp
import numpy as np
import optax
from jax import config
from numpy.polynomial.hermite import hermgauss
import os 
from flows.basis.basis import Basis
import matplotlib.pyplot as plt 
from numpy.polynomial.hermite import hermgauss
from flows.Bases import Hermite 
from flows.types import *
from flows.bases import *
from flows.models.linear import Linear
import math 
import pickle 
from flows.models.iresnet import IResNet, ActivationFunction


## Set up potential and KEO 

In [2]:
mO, mH = [15.994914619257,1.00782503223]
m1 = mH
m2 = mO
a_m,D_m,x0_m = 2.1440, 42301, 0.0
# Number of states to compute and optimize
no_states = 23

# Potential energy surface
def potential(x):
    return (D_m * (1.0 - jnp.exp(-a_m * (x - x0_m)))**2)[:,0]#.reshape(-1,1)

G_to_invcm = (
    constants.value("Planck constant")
    * constants.value("Avogadro constant")
    * 1e16
    / (4.0 * np.pi**2 * constants.value("speed of light in vacuum"))
    * 1e5
)

def abs_det_jac_x(model, params, x):
    def det(params):
        def det(x):
            return jnp.abs(jnp.linalg.det(jax.jacrev(model.apply, 1)(params, x)))
        return jax.vmap(det, in_axes=0)(x)
    return jax.jit(det)(params)
# G-matrix for the Morse potential
def gmat(x):
    npoints = x.shape[0]
    gvib = (m1+m2)/(m1*m2) * jax.numpy.ones_like(x)[:,:,None] * G_to_invcm
    grot = jnp.zeros((npoints,))
    gcor = jnp.zeros((npoints, 1))
    return gvib, grot, gcor



## Set up basis and quadrature grid 

In [3]:
x, w = hermgauss(100)
x = x.reshape(-1,1)
nmax = 22
n_basis = [nmax for _ in range(x.shape[1])]
w_basis = [1 for _ in range(x.shape[1])]
# Note that when we project we need to multiply by the weights of the basis
basis_no = Hermite.init_basis(n_basis, w_basis, nmax, orthotype=orthoType.n_ortho) # weights for generating non-orthonormal basis
basis_pr = Hermite.init_basis(n_basis, w_basis, nmax, orthotype=orthoType.proj) # 
#weights for generating the basis on which we project
basis_r = Hermite.init_basis(n_basis, w_basis, nmax, orthotype=orthoType.ortho) # 
psi_no = partial(Hermite.batch_basis_values, basis_no)
psi_pr = partial(Hermite.batch_basis_values, basis_pr)

psi_o = partial(Hermite.batch_basis_values, basis_r)  

def psi_pr(x):
    y = psi_no(x)
    n_basis = y.shape[1]
    for n in range(n_basis):
        y = y.at[:,n].set(y[:,n] * 1./((math.factorial(n)*2**n)*np.sqrt(jnp.pi)))
    return y

dpsi_o = partial(Hermite.batch_dbasis_values, basis_r)

## Set up a normalizing flow

In [ ]:
a = jnp.array([1.43630802])
b = jnp.array([-1.4624287])

nblocks = 5
xmin = np.min(x, axis=0)
xmax = np.max(x, axis=0)
    
model = IResNet(
        a=a,
        b=b,
        intervals=jnp.array([[-jnp.inf, jnp.inf]]),
        xmin=xmin,
        xmax=xmax,
        features=[8, 8, 1],
        activations=[
                ActivationFunction.LIPSWISH,
                ActivationFunction.LIPSWISH,
                ActivationFunction.LIPSWISH, #
                ],
        no_resnet_blocks=nblocks)

model_r = lambda params, x: model.apply(params, x[:,0].reshape(-1,1), inverse=True)

model_x = lambda params, x: (
    model.apply(params, x[:, 0].reshape(-1, 1), inverse=False)
    if x.ndim == 2 else
    model.apply(params, jnp.array(x[0]).reshape(1, -1), inverse=False)[0, :]
    )

params = model.init(jax.random.PRNGKey(0), x[:,0].reshape(-1,1))

r = model_r(params, x)
rmin = np.min(r, axis=0)
rmax = np.max(r, axis=0)
# loss and test functions

def _grad_log_abs_det_jac_x(model, params, x_batch, **kwargs):
    def det(x):
        return jnp.log(
            jnp.abs(jnp.linalg.det(jax.jacrev(model, argnums=1)(params, x, **kwargs)))
        )

    return jax.vmap(jax.grad(det), in_axes=0)(x_batch)

def _jac_x(model, params, x_batch, **kwargs):
    def jac(x):
        return jax.jacrev(model, argnums=1)(params, x, **kwargs)

    return jax.vmap(jac, in_axes=0)(x_batch)



In [5]:
def hamiltonian(params, x, psi1, dpsi1, psi2, dpsi2, w):
    r = model_r(params, x)
    p = potential(r)
    ovlp = jnp.zeros(p.shape)
    gvib, _, _ = gmat(r)
    dlog_det = _grad_log_abs_det_jac_x(model_x, params, r)
    
    df = _jac_x(model_x, params, r)
    gvib1 = jnp.einsum("gka,gab...,glb->gkl...", df, gvib, df)
    gvib2 = 0.5 * jnp.einsum("gka,gab...,gb->gk...", df, gvib, dlog_det)
    gvib3 = 0.5 * jnp.einsum("ga,gab...,gkb->gk...", dlog_det, gvib, df)
    gvib4 = 0.25 * jnp.einsum("ga,gab...,gb->g...", dlog_det, gvib, dlog_det)

    print("Shapes in hamiltonian: ", dpsi1.shape, gvib1.shape, dpsi2.shape, w.shape)
    #exit()
    keo_vib = (
            jnp.einsum("gik,gkl...,gjl,g->ij...", dpsi1, gvib1, dpsi2, w)
            + jnp.einsum("gik,gk...,gj,g->ij...", dpsi1, gvib2, psi2, w)
            + jnp.einsum("gi,gk...,gjk,g->ij...", psi1, gvib3, dpsi2, w)
            + jnp.einsum("gi,g...,gj,g->ij...", psi1, gvib4, psi2, w)
        )
    poten = jnp.einsum("gi,gj,g...,g->ij...", psi1, psi2, p, w)
    overlap = jnp.einsum("gi,gj,g...,g->ij...", psi1, psi2, ovlp, w)
    ham = 0.5 * (keo_vib ) + poten
    return ham

def eigensolve(par, psi1, dpsi1, psi2, dpsi2, no_states):
    h = hamiltonian(par, x, psi1, dpsi1, psi2, dpsi2, w)
    e, v = jax.numpy.linalg.eigh(h)
    return e[:no_states], v

eigensolve_jit = partial(jax.jit, static_argnums=(5,))(eigensolve)

def loss_fn(par, psi_1, dpsi_1, psi_2, dpsi_2, no_states):
    e, _ = eigensolve(par, psi_1, dpsi_1, psi_2, dpsi_2, no_states)
    return jnp.sum(e[:no_states])

def loss_grad_fn(par, psi_1, dpsi_1, psi_2, dpsi_2, no_states):
    return jax.value_and_grad(loss_fn)(
        par,
        psi_1,
        dpsi_1,
        psi_2,
        dpsi_2,
        no_states   
    )

_psi = psi_o(x)
_dpsi_ = dpsi_o(x)
print("First few eigenvalues on a test set using initial params:")
e, _ = eigensolve_jit(params, _psi, _dpsi_, _psi, _dpsi_, no_states)
print('Loss',jnp.sum(e))
print(e[0], e[:10] - e[0])
    

First few eigenvalues on a test set using initial params:
Shapes in hamiltonian:  (100, 23, 1) (100, 1, 1) (100, 23, 1) (100,)
Loss 1985056.9893237082
1838.9728877743732 [    0.          3555.34614676  6947.22612184 10175.63992569
 13240.58756729 16142.06940494 18880.15194147 21457.55168175
 23916.17566389 26456.78150468]


In [6]:
# optimisation
optx = optax.adam(learning_rate=0.001)
opt_state = optx.init(params)

@partial(jax.jit, static_argnums=(2,))
def update_params(params, opt_state, no_states):
    loss_val, grad = loss_grad_fn(params, _psi, _dpsi_, _psi, _dpsi_, no_states)
    updates, opt_state = optx.update(grad, opt_state)
    params = optax.apply_updates(params, updates)
    return loss_val, params, opt_state

for i in range(0,20001): 
    loss_val, params, opt_state = update_params(params, opt_state, no_states)
    #print(i, loss_val)

    if i % 30 == 0:
        e, v = eigensolve_jit(params, _psi, _dpsi_, _psi, _dpsi_, no_states)
        
        loss_val = np.sum(e[:no_states])
        print("Test loss:", loss_val, "First few eigenvalues:", e[:10])
with open("simulations_data/Morse_NF_Hermite.pkl", "wb") as f:
    pickle.dump({
        "params": params,
        "eigenvalues": e,
        "eigenvectors": v
    }, f)


Shapes in hamiltonian:  (100, 23, 1) (100, 1, 1) (100, 23, 1) (100,)
Test loss: 1812732.1875069027 First few eigenvalues: [ 1838.97288777  5394.31903454  8786.19900976 12014.61281422
 15079.56045275 17981.04220402 20719.09393027 23295.48896551
 25740.11786776 28223.02372208]
Test loss: 795766.8030222637 First few eigenvalues: [ 1838.97324337  5394.32202737  8786.22151877 12014.78480069
 15080.42952483 17983.87990599 20725.51598101 23304.70813024
 25720.46182891 27973.10889889]
Test loss: 768715.2710243447 First few eigenvalues: [ 1838.97466851  5394.34548068  8786.49090434 12016.33266116
 15085.34761005 17993.65419518 20739.70253007 23325.3290135
 25759.37274038 28050.05374775]
Test loss: 749116.8563390714 First few eigenvalues: [ 1838.98105132  5394.38582641  8786.51984466 12016.5101357
 15088.49768425 18007.87743042 20773.5208212  23376.86524632
 25814.89133533 28098.31785128]
Test loss: 738741.6149786485 First few eigenvalues: [ 1838.99698882  5394.5017264   8787.22410492 12020.9137

In [7]:
N_values = np.arange(22, 32, 2)
e_values = []
for nmax in N_values:
    
    n_basis = [nmax for _ in range(x.shape[1])]
    w_basis = [1 for _ in range(x.shape[1])]
    # Note that when we project we need to multiply by the weights of the basis
    basis_r = Hermite.init_basis(n_basis, w_basis, nmax, orthotype=orthoType.ortho) # 

    psi_o = partial(Hermite.batch_basis_values, basis_r)  
        
    dpsi_o = partial(Hermite.batch_dbasis_values, basis_r)
    _psi = psi_o(x)
    _dpsi = dpsi_o(x)
    print(type(_psi), _psi.shape)
    print("First few eigenvalues on a test set using initial params:")
    e, v = eigensolve_jit(params, _psi, _dpsi, _psi, _dpsi, no_states)
    e_values.append(e)
np.savez(f"simulations_data/Morse_NF_Hermite.npz", N_values=N_values, e_values=e_values, eigenvectors=v)

<class 'jaxlib._jax.ArrayImpl'> (100, 23)
First few eigenvalues on a test set using initial params:
<class 'jaxlib._jax.ArrayImpl'> (100, 25)
First few eigenvalues on a test set using initial params:
Shapes in hamiltonian:  (100, 25, 1) (100, 1, 1) (100, 25, 1) (100,)
<class 'jaxlib._jax.ArrayImpl'> (100, 27)
First few eigenvalues on a test set using initial params:
Shapes in hamiltonian:  (100, 27, 1) (100, 1, 1) (100, 27, 1) (100,)
<class 'jaxlib._jax.ArrayImpl'> (100, 29)
First few eigenvalues on a test set using initial params:
Shapes in hamiltonian:  (100, 29, 1) (100, 1, 1) (100, 29, 1) (100,)
<class 'jaxlib._jax.ArrayImpl'> (100, 31)
First few eigenvalues on a test set using initial params:
Shapes in hamiltonian:  (100, 31, 1) (100, 1, 1) (100, 31, 1) (100,)
